# Repeat task 9 training on the same GPU model

**Our next question: do two runs agree more closely when both use an A100 GPU?**

Our first pair used two different GPU models: Blackwell for A and A100 for B.
Task 0 ended at 29.2% in A and 36.4% in B. The recorded calculations first differed
at training step 1. Different GPU models are one possible cause, but we have not
proved that they explain the difference.

Today we will repeat the pair with **A100 in both runs**. We will keep the starting
model, images, seed, and training settings the same. The notebook will check the
GPU name and stop before training if the wrong model is connected.

Use a new experiment folder for this pair. Keep the first pair's files so we can
compare the findings later.

You will use this notebook on three occasions:

1. Train task 9 once and save the results. We call this **run A**.
2. Start a new Colab computer, repeat the training, and save the results.
   We call this **run B**.
3. Compare the saved results from A and B.

The notebook records the details for you. You only need to choose which of these
three actions to perform and follow the instructions below.

### Words you will see

- **Model:** the system we train to recognize images.
- **Task:** a group of image classes the model learns to recognize. Task 9 is one
  such group. We also test task 0 to see whether learning task 9 changes an older
  ability.
- **Checkpoint:** a file containing a saved model and information needed to
  continue training it. Our starting file was saved after learning tasks 3 and 0,
  then applying the forgetting operation to task 3. The old name `post-U3`
  means "after the forget-task-3 operation."
- **Colab session:** the time you use one computer provided by Colab. Starting a
  new session gives us a new computer environment for the second run. Reopening
  the notebook page alone does not necessarily give us a new session.
- **GPU:** a processor that helps train the model. Choose a GPU for A and B.
  The comparison can use the regular processor, called a **CPU**.
- **Code cell:** a box containing Python instructions. Press its play button to
  run it. Wait until it finishes before running the next box.


## 1. Choose what to do and connect Google Drive

For your first run:

1. Choose **A100** in Colab's runtime settings. A100 is the GPU model required
   for this experiment. If A100 is not available, do not substitute another model.
2. In the next code cell, leave `MODE = "A"`. This tells the notebook to do run A.
3. Look at `EXPERIMENT_ID`. This is the name of the folder for this experiment.
   If you have already used that name, give this new experiment a different name,
   such as `20261005_L9_same_gpu_02`.
4. Keep that exact name for run B and the comparison. Write it down.
5. Check `CHECKPOINT`. It is the location of our starting model file on Drive.
   Change it only if your file is stored somewhere else.
6. Leave `REQUIRED_GPU` unchanged. It contains the exact name of the A100
   model used in your earlier run B.
7. Leave the other values unchanged and run the cell. Colab will ask you to
   connect your Google Drive account. Follow the instructions it shows.

Then run the second code cell below. It downloads and installs our experiment
code. When it finishes, it prints **Code version:** followed by a long identifier.
That identifier records which version of the code we used. You do not need to
edit it. The notebook will select that same version for run B.

Later, change only `MODE` to choose the next action:

| Value | What the notebook does | Processor to choose |
| --- | --- | --- |
| `"A"` | First training run | A100 GPU |
| `"B"` | Second training run on a new Colab computer | A100 GPU |
| `"COMPARE"` | Compare the two saved runs | CPU |

If a cell shows an error, stop there. Copy the error so we can explain and fix it.
Running later cells will not repair an earlier failed step.


In [ ]:
from pathlib import Path
import json
from google.colab import drive

drive.mount("/content/drive")
MODE = "A"  # A = first run; B = second run; COMPARE = read both results
EXPERIMENT_ID = "20261005_L9_same_gpu_01"
REQUIRED_GPU = "NVIDIA A100-SXM4-40GB"  # Keep the same GPU model for both runs.
REVISION = "main"  # Leave this unchanged. B uses the code version saved by A.
CHECKPOINT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/full_sequence_27step_candidate/checkpoint_seq1_resnet50_seed0_after_request02_forget3.pt")
DATA = Path("/content/data/tiny-imagenet-200")
EXPERIMENT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/L9_session_replay") / EXPERIMENT_ID
CONTRACT = EXPERIMENT / "contract.json"
assert MODE in {"A", "B", "COMPARE"}
if MODE != "A":
    assert CONTRACT.is_file(), "The first run's settings file was not found. Check EXPERIMENT_ID and complete run A first."
    REVISION = json.loads(CONTRACT.read_text())["code"]["commit"]
else:
    assert not EXPERIMENT.exists(), "This experiment folder already exists. For a new run A, change EXPERIMENT_ID to a new name."


In [ ]:
# Download the experiment code. Run this before the training cell.
import os
import subprocess
import sys
REPO = Path("/content/uncle-L9-replay")
assert not any(name == "uncle" or name.startswith("uncle.") for name in sys.modules), "The training code is already loaded. Start a new Colab session before running setup again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "This folder does not contain the expected experiment code. Start a new Colab session."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "This folder contains code from a different project. Start a new Colab session."
    status = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    assert not status, "Code files were changed in this session. Save your changes, then start a new Colab session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
selected = "origin/main" if REVISION == "main" else REVISION
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", selected], check=True)
assert (REPO / "uncle/replay.py").is_file(), "Selected revision does not contain the replay workflow. Use a published revision containing notebook 11 and uncle/replay.py."
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Code version:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Train task 9

Run the next cell and wait for it to finish. It first prints the GPU name and
checks that it is the required A100 model. It may then download the images.
The progress bar then shows the training progress.

The notebook loads the starting model and trains it to recognize task 9 images.
It also measures task 0, an older task. This tells us whether the new training
changes an ability the model already had.

During training, the model changes its internal numbers many times. We call
each change a **training step**. The notebook records the image order, the
training error, and a summary of the model's numbers at each step. These records
will help us find where A and B first become different.

When training finishes, the cell prints **Training finished. Results saved in:**
followed by a folder ending in `A` or `B`. Run step 3 next.


In [ ]:
if MODE in {"A", "B"}:
    # Set one GPU calculation option to match run A before loading the training software.
    if MODE == "B":
        workspace = json.loads(CONTRACT.read_text())["settings"]["cublas_workspace_config"]
        if workspace is None:
            os.environ.pop("CUBLAS_WORKSPACE_CONFIG", None)
        else:
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = workspace
    import torch
    from uncle import checkpoint as checkpointing
    from uncle.config import Config
    from uncle.streams import build_tasks
    from uncle.replay import check_gpu_model, run_replay

    assert torch.cuda.is_available(), "Training needs a GPU. Choose GPU in Colab's runtime settings and start again."
    first_gpu = None
    if MODE == "B":
        first_gpu = json.loads(CONTRACT.read_text()).get("gpu_model")
        assert first_gpu is not None, "Run A did not check the GPU model. Use a new experiment name and start with A."
    gpu_name = check_gpu_model(REQUIRED_GPU, first_run_gpu=first_gpu)
    print("GPU checked:", gpu_name)
    assert CHECKPOINT.is_file(), f"Starting model file not found: {CHECKPOINT}. Check its location on Drive."
    # Give run B the same calculation settings as run A.
    if MODE == "B":
        settings = json.loads(CONTRACT.read_text())["settings"]
        torch.use_deterministic_algorithms(settings["deterministic_algorithms"],
                                          warn_only=settings["deterministic_warn_only"])
        torch.backends.cudnn.benchmark = settings["cudnn_benchmark"]
        torch.backends.cudnn.deterministic = settings["cudnn_deterministic"]
        torch.set_float32_matmul_precision(settings["float32_matmul_precision"])
        torch.backends.cudnn.allow_tf32 = settings["cudnn_allow_tf32"]
        torch.backends.cuda.matmul.allow_tf32 = settings["matmul_allow_tf32"]
    payload = checkpointing.load(CHECKPOINT)
    config = Config(**payload["config"])
    del payload
    assert config.dataset == "tiny_imagenet" and config.backbone == "resnet50"
    tasks = build_tasks(config, root=DATA, include=("0", "3", "9"), download=True)
    EXPERIMENT.mkdir(parents=True, exist_ok=True)
    RUN_FOLDER = run_replay(CHECKPOINT, tasks, EXPERIMENT / MODE, CONTRACT, label=MODE, required_gpu=REQUIRED_GPU)
    print("Training finished. Results saved in:", RUN_FOLDER)


## 3. Check the saved files and finish this GPU session

Run the next cell. It checks that the saved files can be read and contain all
expected training steps. Then it finishes writing to Drive and disconnects the
Colab computer. Disconnecting frees the GPU you were using.

You should see **Saved files checked:** and then **Drive saved. Ending this GPU
session.** The notebook page may remain open after the computer disconnects.
Your results stay on Drive.

If a check fails, stop and copy the error. Do not delete the saved results.

If training was interrupted before finishing, the partial results are still
useful to inspect. To start the experiment again, choose a new `EXPERIMENT_ID`
and begin with run A. This notebook does not continue an interrupted run.


In [ ]:
if MODE in {"A", "B"}:
    from uncle.replay import verify_run
    from google.colab import runtime
    rows = verify_run(RUN_FOLDER)
    print("Saved files checked:", len(rows) - 1, "training steps recorded in", RUN_FOLDER, flush=True)
    drive.flush_and_unmount(timeout_ms=600000)
    print("Drive saved. Ending this GPU session.", flush=True)
    runtime.unassign()


## 4. Do run B on a new Colab computer

After run A ends its session:

1. Connect to a new Colab session with an **A100 GPU**.
2. At the top of this notebook, change `MODE = "A"` to `MODE = "B"`.
3. Keep the same `EXPERIMENT_ID`, `CHECKPOINT`, and `REQUIRED_GPU` values
   you used for A.
4. Run the code cells in steps 1, 2, and 3, in that order.

Run B starts from the original saved model, just as A did. It does not start
from the model produced by A. This makes the two runs comparable.

The notebook checks that the code, starting file, image files, and settings
match A. It also checks the GPU model and the recorded training-software
versions. It saves B's results in a separate folder. Step 3 then ends B's GPU
session.

## 5. Compare A and B

After run B finishes:

1. Connect to a new Colab session. Choose CPU for this part.
2. Change the value at the top to `MODE = "COMPARE"`.
3. Keep the same `EXPERIMENT_ID`.
4. Run both code cells in step 1 to connect Drive and load the experiment code.
5. Skip the training cells in steps 2 and 3. Run the comparison cell below.

This reads the saved results. It does not train the model again.

The comparison shows each run's final **accuracy**: the percentage of test images
it classified correctly. For example, 40% accuracy means 40 correct answers out
of 100 images.

It also tells you whether the recorded training details matched, or which
training step first differed. The full report is saved as `comparison.json`
in your experiment folder on Drive. You can share that file with me so we can
read the result together.


In [ ]:
if MODE == "COMPARE":
    from uncle.replay import compare_replays
    report = compare_replays(EXPERIMENT / "A", EXPERIMENT / "B",
                             EXPERIMENT / "comparison.json")
    print("Final accuracy (percentage of test images answered correctly):")
    for label, scores in report["final_accuracies"].items():
        print(f"Run {label}: task 0 = {scores['0']:.1f}%; task 9 = {scores['9']:.1f}%")
    difference = report["first_trace_difference"]
    names = {
        "indices": "order of the training images",
        "input_sha256": "image and label values given to the model",
        "loss": "training error",
        "parameters_sha256": "numbers stored in the model",
        "buffers_sha256": "extra values the model keeps during training",
        "rng_sha256": "position in the random-choice sequence",
        "new_task_embedding_sha256": "starting task-9 code (numbers used to identify task 9)",
        "starting_accuracies": "test scores before training",
        "epoch": "number of passes through the training images",
        "trace_length": "number of recorded training steps",
    }
    if difference is None:
        print("All recorded training details matched.")
    else:
        where = "before the first training update" if difference["step"] == 0 else f"at training step {difference['step']}"
        print("First recorded difference:", where)
        for field in difference["fields"]:
            print(" - Different", names.get(field, field))
    if report["metadata_differences"]:
        print("Some computer, software, or setup details also differed. These are saved in the report.")
    if difference is None and report["final_accuracies"]["A"] == report["final_accuracies"]["B"]:
        print("This pair did not reproduce the earlier problem. We have not yet explained the older differences.")
    else:
        print("A difference was found. Read the report before choosing the next check.")
    print("Full report saved in:", EXPERIMENT / "comparison.json")
    assert json.loads((EXPERIMENT / "comparison.json").read_text()) == report
    drive.flush_and_unmount(timeout_ms=600000)


## What does the result tell us?

If the two runs differ, we have evidence about where to investigate next.
The first difference helps us choose the next check; it does not automatically
explain the cause.

For example, different image orders suggest we should check how images were
selected. Different model numbers after the same images suggest we should check
the calculations and computer settings.

If both runs match, or their final scores are closer than the earlier pair's
scores, that supports further investigation of the GPU difference. One pair does
not prove that the GPU was the cause. If the new runs still differ, we must look
for other causes even when the GPU model matches.

Once we understand the repeated-run problem, the next experiment will train
several tasks in order, without asking the model to forget any task. We will
measure older tasks after each new task is learned. That will help us study why
older abilities fall toward chance accuracy.

### Details recorded automatically

Training uses some random choices. A **seed** is a number used to start a
repeatable sequence of random choices. Both runs use the same seed.
The starting file also contains the saved position in PyTorch's random sequence.
**PyTorch** is the software that trains our model. Both runs restore that saved
position, rather than only starting the sequence again from the seed.

The old starting file does not contain the random states for two other tools,
Python and NumPy. For this experiment, both runs start those tools from the same
seed. This is a recorded choice for today's experiment; we do not know whether
all older runs made the same choice.

The notebook keeps a settings record in `contract.json`. Here, "contract" is
just the filename for the record used to check that B matches A.
It also stores short digital fingerprints of files and model numbers. A
fingerprint is called a **hash**. Comparing hashes lets us check whether the
recorded contents are identical without printing all of them. Creating these
records adds time, so today's run time includes both training and recording.
